## Join the UNIONS photometry catalogs into a single (hdf5) file

In [1]:
import fitsio as fio
import h5py 
import os
import time
import numpy as np 
import gc
import datetime

In [2]:
#paths
base_path = "/arc/projects/unions/catalogues/unions/GAaP_photometry/UNIONS5000/"
tile_list = os.listdir(base_path)
cat_file = "{tile}_ugriz_photoz_ext.cat"

today = str(datetime.date.today())
outfile = "./UNIONS5000_"+cat_file.format(tile=str(datetime.date.today())) + ".h5"

print(f"{len(tile_list)} tiles found")

print(f'We will output the concatenated file to {outfile}')


20859 tiles found
We will output the concatenated file to ./UNIONS5000_2025-04-24_ugriz_photoz_ext.cat.h5


In [3]:
#load one fits file for the header info
f_example = fio.FITS(base_path +"/"+ "UNIONS.316.239" +"/"+ cat_file.format(tile="UNIONS.316.239"))
example_dtype = f_example[1].get_rec_dtype()

In [7]:
# edit this if you want different columns

cols_basic = [
    "ALPHA_J2000", "DELTA_J2000", "Flag", 
]

cols_mag = ["MAG_GAAP{ap}_{band}", "MAGERR_GAAP{ap}_{band}", "MAG_LIM_{band}", "MAG_AUTO",
           "FLAG_GAAP{ap}_{band}",
           ]

cols_z = ["Z_B", "Z_B_MIN", "Z_B_MAX"]

cols = []
cols += cols_basic
for ap in ["", "_0p7", "_1p0"]: #which Gaap apatures do you want
    for band in ["u","g","r", "i", "z","z2"]:
        for c in cols_mag:
            cols += [c.format(ap=ap, band=band)]
cols += cols_z

print(f'Selecting these columns: {cols}')
print(len(cols))

#not to self: add column for tile


Selecting these columns: ['ALPHA_J2000', 'DELTA_J2000', 'Flag', 'MAG_GAAP_u', 'MAGERR_GAAP_u', 'MAG_LIM_u', 'MAG_AUTO', 'FLAG_GAAP_u', 'MAG_GAAP_g', 'MAGERR_GAAP_g', 'MAG_LIM_g', 'MAG_AUTO', 'FLAG_GAAP_g', 'MAG_GAAP_r', 'MAGERR_GAAP_r', 'MAG_LIM_r', 'MAG_AUTO', 'FLAG_GAAP_r', 'MAG_GAAP_i', 'MAGERR_GAAP_i', 'MAG_LIM_i', 'MAG_AUTO', 'FLAG_GAAP_i', 'MAG_GAAP_z', 'MAGERR_GAAP_z', 'MAG_LIM_z', 'MAG_AUTO', 'FLAG_GAAP_z', 'MAG_GAAP_0p7_u', 'MAGERR_GAAP_0p7_u', 'MAG_LIM_u', 'MAG_AUTO', 'FLAG_GAAP_0p7_u', 'MAG_GAAP_0p7_g', 'MAGERR_GAAP_0p7_g', 'MAG_LIM_g', 'MAG_AUTO', 'FLAG_GAAP_0p7_g', 'MAG_GAAP_0p7_r', 'MAGERR_GAAP_0p7_r', 'MAG_LIM_r', 'MAG_AUTO', 'FLAG_GAAP_0p7_r', 'MAG_GAAP_0p7_i', 'MAGERR_GAAP_0p7_i', 'MAG_LIM_i', 'MAG_AUTO', 'FLAG_GAAP_0p7_i', 'MAG_GAAP_0p7_z', 'MAGERR_GAAP_0p7_z', 'MAG_LIM_z', 'MAG_AUTO', 'FLAG_GAAP_0p7_z', 'MAG_GAAP_1p0_u', 'MAGERR_GAAP_1p0_u', 'MAG_LIM_u', 'MAG_AUTO', 'FLAG_GAAP_1p0_u', 'MAG_GAAP_1p0_g', 'MAGERR_GAAP_1p0_g', 'MAG_LIM_g', 'MAG_AUTO', 'FLAG_GAAP_1p0_g', 

# Here we will loop through the files to count the number of rows in each, this is not an efficient algorithm, but allows us to create the correct sized hdf5 datasets later. 

In [5]:
# count the number of rows in each file
# (and also experiment with how long it takes to get the files from disk vs memory)
deltas = []
nrows = []
missing_tiles = []
for i, tile in enumerate(tile_list):
    if i%200 == 0 and i!=0 :
        print(f"{i}/{len(tile_list)} approx time remaining {time_left}s, mean of last 100 nrows {np.round(np.mean(nrows[-99:]),1)}")
    
    s = time.time()
    filename = cat_file.format(tile=tile)
    filepath = base_path +"/"+ tile +"/"+ filename

    #check that file exists
    if not os.path.exists(filepath):
        print(f"tile {tile} has no catalog")
        nrows.append(0)
        missing_tiles.append(tile)
        continue

    with fio.FITS(filepath) as f:
        nrows.append(f[1].get_nrows())

    e = time.time()
    delta = e-s
    deltas.append(delta)
    time_left = np.round((len(tile_list)-i)*np.mean(deltas),1)

    

200/20859 approx time remaining 1890.0s, mean of last 100 nrows 31391.0
400/20859 approx time remaining 1887.3s, mean of last 100 nrows 31979.6
600/20859 approx time remaining 1850.0s, mean of last 100 nrows 31585.4
800/20859 approx time remaining 1810.0s, mean of last 100 nrows 32093.9
1000/20859 approx time remaining 1794.6s, mean of last 100 nrows 31823.5
1200/20859 approx time remaining 1763.9s, mean of last 100 nrows 32073.3
1400/20859 approx time remaining 1731.3s, mean of last 100 nrows 32411.7
1600/20859 approx time remaining 1705.1s, mean of last 100 nrows 31689.1
1800/20859 approx time remaining 1677.3s, mean of last 100 nrows 30925.6
tile UNIONS.366.277 has no catalog
tile UNIONS.191.248 has no catalog
2000/20859 approx time remaining 1663.3s, mean of last 100 nrows 33507.0
tile UNIONS.133.307 has no catalog
2200/20859 approx time remaining 1636.8s, mean of last 100 nrows 30700.4
2400/20859 approx time remaining 1616.8s, mean of last 100 nrows 31894.8
tile UNIONS.665.186 has

## Save a list of tiles that dont have the requested catalog

In [6]:
missing_tile_file = open(cat_file.format(tile="missing_tiles")+'.txt', 'w')
missing_tile_file.write('\n'.join(missing_tiles))
missing_tile_file.close()
ntot = np.sum(nrows)
print(f'{ntot} total objects across all catalogs')

674819292 total objects across all catalogs


## Now we can actually transfer the data from the individual tile catalogs into our large hdf5 file

In [7]:

with h5py.File(outfile, "w") as output:

    #make an empty hdf5 group to save the catalogs to 
    grp = output.create_group("catalog")
    
    #make empty data sets for each column
    for c in cols:
        col_dtype = example_dtype[0][c] #get from f_example
        grp.create_dataset(c, shape=(ntot,), dtype=col_dtype)        # 64-bit float

    deltas = [] #for timing info
    for i, tile in enumerate(tile_list):
        if i%200 == 0 and i!=0 :
            print(f"{i}/{len(tile_list)} approx time remaining {time_left}s, mean of last 100 nrows {np.round(np.mean(nrows[-99:]),1)}")
        
        s = time.time()
        filename = cat_file.format(tile=tile)
        filepath = base_path +"/"+ tile +"/"+ filename
    
        #check that file exists
        if tile in missing_tiles:
            print(f"tile {tile} has no catalog")
            continue

        start = sum(nrows[:i])
        end = sum(nrows[:i+1])
    
        with fio.FITS(filepath) as f:
            for c in cols:
                grp[c][start:end] = f[1][c].read()
    
        e = time.time()
        delta = e-s
        deltas.append(delta)
        time_left = np.round((len(tile_list)-i)*np.mean(deltas),1)
    
        #break
            


200/20859 approx time remaining 9026.0s, mean of last 100 nrows 31777.4
400/20859 approx time remaining 9104.3s, mean of last 100 nrows 31777.4
600/20859 approx time remaining 8806.4s, mean of last 100 nrows 31777.4
800/20859 approx time remaining 8731.1s, mean of last 100 nrows 31777.4
1000/20859 approx time remaining 8673.1s, mean of last 100 nrows 31777.4
1200/20859 approx time remaining 8501.7s, mean of last 100 nrows 31777.4
1400/20859 approx time remaining 8340.9s, mean of last 100 nrows 31777.4
1600/20859 approx time remaining 8163.7s, mean of last 100 nrows 31777.4
1800/20859 approx time remaining 8040.8s, mean of last 100 nrows 31777.4
tile UNIONS.366.277 has no catalog
tile UNIONS.191.248 has no catalog
2000/20859 approx time remaining 7966.7s, mean of last 100 nrows 31777.4
tile UNIONS.133.307 has no catalog
2200/20859 approx time remaining 8017.0s, mean of last 100 nrows 31777.4
2400/20859 approx time remaining 8088.1s, mean of last 100 nrows 31777.4
tile UNIONS.665.186 has